In [28]:
#!pip install tqdm
import glob
import pandas as pd
import re
import requests
import time
from tqdm import tqdm

In [19]:
df = pd.read_csv("MAVEable_final_is_feasible.csv")
df.columns.to_list
cols = ["GeneSymbol", "#AlleleID", "GeneID", "HGNC_ID", "Entry", "Length", "Ensembl", "Benign", "Conflicting classifications of pathogenicity", "Likely benign", "Likely pathogenic", "Pathogenic", "Uncertain significance", "location_feasible"]
subset = df[cols]
MAVEable_filtered = subset[(subset["location_feasible"] == True)]
MAVEable_filtered

,GeneSymbol,#AlleleID,GeneID,HGNC_ID,Entry,Length,Ensembl,Benign,Conflicting classifications of pathogenicity,Likely benign,Likely pathogenic,Pathogenic,Uncertain significance,location_feasible
0,AAAS,20083,8086,HGNC:13666,Q9NRG9,546.0,ENST00000209873.9 [Q9NRG9-1];ENST00000394384.7...,0,6,3,8,4,109,True
1,AAGAB,703299,79719,HGNC:25662,Q6PD74,315.0,ENST00000261880.10 [Q6PD74-1];ENST00000542650....,3,4,3,1,0,46,True
2,AARS1,23505,16,HGNC:20,P49588,968.0,ENST00000261772.13 [P49588-1];ENST00000674691....,2,43,21,7,4,693,True
4,AASS,247012,10157,HGNC:17366,Q9UDR5,926.0,ENST00000393376.5;ENST00000417368.7;ENST000006...,4,2,5,2,0,103,True
7,ABCA12,17894,26154,HGNC:14637,Q86UK0,2595.0,ENST00000272895.12 [Q86UK0-1];ENST00000389661....,22,74,90,22,2,255,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3901,ZNF462,225858,58499,HGNC:21684,Q96JM2,2506.0,ENST00000277225.10 [Q96JM2-1];ENST00000441147....,7,6,61,2,0,299,True
3902,ZNF469,132429,84627,HGNC:23216,Q96JG9,3953.0,ENST00000565624.3;,17,370,237,1,0,1967,True
3905,ZNF711,209308,7552,HGNC:13128,Q9Y462,761.0,ENST00000276123.7 [Q9Y462-1];ENST00000360700.4...,1,3,7,1,0,84,True
3912,ZSWIM6,165524,57688,HGNC:29316,Q9HCJ5,1215.0,ENST00000252744.6;,31,29,46,2,0,338,True


In [20]:
import io, gzip, urllib.request

# ── Download the MANE Select summary from NCBI FTP ──────────────────────────
# Auto-detect the current versioned filename from the directory listing
ftp_base = "https://ftp.ncbi.nlm.nih.gov/refseq/MANE/MANE_human/current/"
with urllib.request.urlopen(ftp_base) as resp:
    listing = resp.read().decode()

m = re.search(r'(MANE\.GRCh38\.v[\d.]+\.summary\.txt\.gz)', listing)
if not m:
    raise FileNotFoundError("Could not locate MANE summary file in NCBI FTP listing.")
mane_filename = m.group(1)
mane_url = ftp_base + mane_filename
print(f"Using MANE file: {mane_filename}")

with urllib.request.urlopen(mane_url) as resp:
    mane_df = pd.read_csv(
        io.BytesIO(resp.read()),
        sep='\t',
        compression='gzip',
        dtype=str
    )

# Keep only MANE Select (exclude MANE Plus Clinical)
mane_select = mane_df[mane_df['MANE_status'] == 'MANE Select'].copy()

# Strip version suffix from Ensembl_nuc (e.g. ENST00000209873.9 → ENST00000209873)
mane_select['mane_enst'] = mane_select['Ensembl_nuc'].str.extract(r'(ENST\d+)')

# Build HGNC_ID → mane_enst lookup  (HGNC_ID in MANE file is already "HGNC:NNNNN")
hgnc_to_mane = mane_select.set_index('HGNC_ID')['mane_enst'].to_dict()

# ── Join to filtered gene list ────────────────────────────────────────────────
MAVEable_filtered = MAVEable_filtered.copy()
MAVEable_filtered['mane_enst'] = MAVEable_filtered['HGNC_ID'].map(hgnc_to_mane)

total    = len(MAVEable_filtered)
has_enst = MAVEable_filtered['mane_enst'].notna().sum()
missing  = total - has_enst
print(f"Genes with MANE Select ENST : {has_enst} / {total}")
print(f"Genes missing MANE Select   : {missing}")
MAVEable_filtered[['GeneSymbol', 'HGNC_ID', 'mane_enst']].head(10)

Using MANE file: MANE.GRCh38.v1.5.summary.txt.gz
Genes with MANE Select ENST : 2200 / 2201
Genes missing MANE Select   : 1


,GeneSymbol,HGNC_ID,mane_enst
0,AAAS,HGNC:13666,ENST00000209873
1,AAGAB,HGNC:25662,ENST00000261880
2,AARS1,HGNC:20,ENST00000261772
4,AASS,HGNC:17366,ENST00000417368
7,ABCA12,HGNC:14637,ENST00000272895
9,ABCA3,HGNC:33,ENST00000301732
10,ABCA4,HGNC:34,ENST00000370225
11,ABCB11,HGNC:42,ENST00000650372
12,ABCB4,HGNC:45,ENST00000649586
17,ABCC6,HGNC:57,ENST00000205557


In [21]:
ENSEMBL_POST_URL = "https://rest.ensembl.org/sequence/id"
HEADERS = {"Content-Type": "application/json", "Accept": "application/json"}
BATCH_SIZE = 50      # Ensembl REST max per POST request
PAUSE_SEC  = 0.15    # ~6-7 req/s, well under the 15 req/s limit


def fetch_cds_batch(enst_ids, batch_size=BATCH_SIZE, pause=PAUSE_SEC, max_retries=3):
    """POST batches of ENST IDs to the Ensembl REST API and return {enst_id: cds_seq}."""
    results = {}
    unique_ids = [i for i in dict.fromkeys(enst_ids) if i is not None]

    for start in tqdm(range(0, len(unique_ids), batch_size), desc="Fetching CDS"):
        batch = unique_ids[start : start + batch_size]
        payload = {"ids": batch, "type": "cds"}

        for attempt in range(1, max_retries + 1):
            resp = requests.post(ENSEMBL_POST_URL, json=payload, headers=HEADERS)
            if resp.status_code == 200:
                for record in resp.json():
                    results[record["id"]] = record.get("seq", "")
                break
            elif resp.status_code == 429:          # rate-limited
                wait = int(resp.headers.get("Retry-After", 5))
                print(f"  Rate-limited – waiting {wait}s …")
                time.sleep(wait)
            else:
                print(f"  Batch {start//batch_size} attempt {attempt}: "
                      f"HTTP {resp.status_code} – {resp.text[:200]}")
                if attempt == max_retries:
                    print(f"  Giving up on batch {start//batch_size}.")
                time.sleep(pause * 5)

        time.sleep(pause)

    return results


enst_ids = MAVEable_filtered['mane_enst'].dropna().tolist()
print(f"Requesting CDS for {len(set(enst_ids))} unique transcripts …")
cds_sequences = fetch_cds_batch(enst_ids)
print(f"\nSuccessfully retrieved: {len(cds_sequences)} sequences")

Requesting CDS for 2198 unique transcripts …


Fetching CDS: 100%|██████████| 44/44 [03:19<00:00,  4.53s/it]


Successfully retrieved: 2198 sequences


In [22]:
MAVEable_filtered['orf_sequence'] = MAVEable_filtered['mane_enst'].map(cds_sequences)

fasta_out = "MAVEable_ORFs.fasta"
written   = 0
missing_seq = []

with open(fasta_out, "w") as fh:
    for _, row in MAVEable_filtered.iterrows():
        seq = row['orf_sequence']
        if pd.notna(seq) and seq:
            header = (f">{row['GeneSymbol']}|{row['HGNC_ID']}|"
                      f"{row['mane_enst']}|{row['Entry']}")
            fh.write(header + "\n")
            # 60-character line wrap
            for i in range(0, len(seq), 60):
                fh.write(seq[i : i + 60] + "\n")
            written += 1
        else:
            missing_seq.append(row['GeneSymbol'])

print(f"Wrote {written} ORF sequences to '{fasta_out}'")
if missing_seq:
    print(f"No CDS retrieved for {len(missing_seq)} genes: {missing_seq[:20]}"
          + (" …" if len(missing_seq) > 20 else ""))

Wrote 2200 ORF sequences to 'MAVEable_ORFs.fasta'
No CDS retrieved for 1 genes: ['LRTOMT']


In [25]:

# ── Locate and load ClinVar SNV file ─────────────────────────────────────────
cv_files = glob.glob("variant_summary*.txt.gz")
if not cv_files:
    raise FileNotFoundError("No variant_summary_*.txt.gz file found in working directory.")
cv_file = sorted(cv_files)[-1]          # use most recent if multiple copies exist
print(f"Loading: {cv_file}")

clinvar_raw = pd.read_csv(cv_file, sep='\t', low_memory=False)

# Replicate the same filtering chain from ClinVar_wrangle_uniprot_gencc_merge.ipynb
cv = clinvar_raw[clinvar_raw['Type'] == 'single nucleotide variant'].copy()
cv = cv[cv['Assembly'] == 'GRCh38']
cv = cv[cv['ClinicalSignificance'].str.contains('pathogenic|benign', case=False, na=False)]
cv = cv[~cv['ClinicalSignificance'].str.contains(
    'penetrance|drug|other|protective|risk|uncertain', case=False, na=False)]
cv = cv[~cv['ReviewStatus'].str.contains('no assertion', case=False, na=False)]

# ── Missense only ─────────────────────────────────────────────────────────────
amino_acid_map = {
    'Ala': 'A', 'Cys': 'C', 'Asp': 'D', 'Glu': 'E', 'Phe': 'F',
    'Gly': 'G', 'His': 'H', 'Ile': 'I', 'Lys': 'K', 'Leu': 'L',
    'Met': 'M', 'Asn': 'N', 'Pro': 'P', 'Gln': 'Q', 'Arg': 'R',
    'Ser': 'S', 'Thr': 'T', 'Val': 'V', 'Trp': 'W', 'Tyr': 'Y',
    'Ter': '*',
}

def extract_p_variant(name):
    m = re.search(r'\(p\.([A-Za-z]{3}\d+[A-Za-z]{3})\)', str(name))
    if not m:
        return ''
    raw = m.group(1)
    return re.sub(r'[A-Z][a-z]{2}', lambda x: amino_acid_map.get(x.group(), x.group()), raw)

cv['p.variant'] = cv['Name'].apply(extract_p_variant)

# Keep only clean missense: [A-Z]\d+[A-Z] with no stop codon
cv_missense = cv[cv['p.variant'].str.fullmatch(r'[A-Z]\d+[A-Z]', na=False)].copy()

# ── Clean classification labels ───────────────────────────────────────────────
sig = cv_missense['ClinicalSignificance'].str.lower()
cv_missense = cv_missense.copy()
cv_missense['class'] = None
cv_missense.loc[sig.str.contains('benign') & ~sig.str.contains('pathogenic'), 'class'] = 'Benign'
cv_missense.loc[sig.str.contains('pathogenic') & ~sig.str.contains('benign'), 'class'] = 'Pathogenic'
cv_missense = cv_missense[cv_missense['class'].notna()]

# Restrict to genes present in MAVEable_filtered
mave_genes = set(MAVEable_filtered['GeneSymbol'])
cv_missense = cv_missense[cv_missense['GeneSymbol'].isin(mave_genes)]

print(f"ClinVar missense variants for MAVEable genes: {len(cv_missense):,}")
print(cv_missense.groupby('class').size())

Loading: variant_summary.txt.gz
ClinVar missense variants for MAVEable genes: 142,241
class
Benign        47233
Pathogenic    95008
dtype: int64


In [26]:
STAR_ORDER = [
    'practice guideline',
    'reviewed by expert panel',
    'criteria provided, multiple submitters, no conflicts',
    'criteria provided, single submitter',
    'criteria provided, conflicting classifications',
]
star_rank = {s: i for i, s in enumerate(STAR_ORDER)}

def rank_review(status):
    s = str(status).lower()
    for i, label in enumerate(STAR_ORDER):
        if label in s:
            return i
    return len(STAR_ORDER)

cv_missense['review_rank'] = cv_missense['ReviewStatus'].apply(rank_review)


def parse_cdna_snv(name):
    """Return (cdna_pos, ref_base, alt_base) from a ClinVar Name string, or (None,None,None)."""
    m = re.search(r'c\.(\d+)([ACGTacgt])>([ACGTacgt])', str(name))
    if m:
        return int(m.group(1)), m.group(2).upper(), m.group(3).upper()
    return None, None, None


def apply_snv(orf_seq, cdna_pos, ref, alt):
    """Substitute one nucleotide in a CDS string (cdna_pos is 1-based). Returns None on mismatch."""
    idx = cdna_pos - 1
    if idx < 0 or idx >= len(orf_seq):
        return None
    if orf_seq[idx].upper() != ref:
        return None
    return orf_seq[:idx] + alt + orf_seq[idx + 1:]


# Build gene → WT CDS lookup from cds_sequences using mane_enst
gene_to_enst = MAVEable_filtered.set_index('GeneSymbol')['mane_enst'].to_dict()
gene_to_wt   = {
    gene: cds_sequences.get(enst)
    for gene, enst in gene_to_enst.items()
    if cds_sequences.get(enst)
}

# ── Sample up to 5 benign + 5 pathogenic per gene and generate mutant ORFs ───
N = 5
records = []
ref_mismatches = 0

for gene, group in tqdm(cv_missense.groupby('GeneSymbol'), desc="Generating mutant ORFs"):
    wt_seq = gene_to_wt.get(gene)
    if not wt_seq:
        continue

    for cls in ('Benign', 'Pathogenic'):
        candidates = (group[group['class'] == cls]
                      .sort_values('review_rank')
                      .drop_duplicates(subset='p.variant'))
        collected = 0
        for _, row in candidates.iterrows():
            if collected >= N:
                break
            pos, ref, alt = parse_cdna_snv(row['Name'])
            if pos is None:
                continue
            mutant = apply_snv(wt_seq, pos, ref, alt)
            if mutant is None:
                ref_mismatches += 1
                continue
            records.append({
                'GeneSymbol':  gene,
                'HGNC_ID':     MAVEable_filtered.loc[MAVEable_filtered['GeneSymbol'] == gene, 'HGNC_ID'].iat[0],
                'mane_enst':   gene_to_enst[gene],
                'p_variant':   row['p.variant'],
                'class':       cls,
                'cdna_change': re.search(r'c\.\S+', str(row['Name'])).group() if re.search(r'c\.\S+', str(row['Name'])) else '',
                'mutant_seq':  mutant,
            })
            collected += 1

variants_df = pd.DataFrame(records)
print(f"Mutant ORFs generated : {len(variants_df)}")
print(f"Ref-base mismatches skipped : {ref_mismatches}")
print(variants_df.groupby('class').size())

Generating mutant ORFs: 100%|██████████| 2174/2174 [00:03<00:00, 651.02it/s]

Mutant ORFs generated : 17919
Ref-base mismatches skipped : 86
class
Benign        8956
Pathogenic    8963
dtype: int64


In [27]:
fasta_out = "MAVEable_missense_variants.fasta"

with open(fasta_out, "w") as fh:
    for _, row in variants_df.iterrows():
        header = (f">{row['GeneSymbol']}|{row['HGNC_ID']}|{row['mane_enst']}"
                  f"|{row['p_variant']}|{row['cdna_change']}|{row['class']}")
        fh.write(header + "\n")
        seq = row['mutant_seq']
        for i in range(0, len(seq), 60):
            fh.write(seq[i : i + 60] + "\n")

print(f"Wrote {len(variants_df)} mutant ORF sequences to '{fasta_out}'")
print(f"  Benign    : {(variants_df['class'] == 'Benign').sum()}")
print(f"  Pathogenic: {(variants_df['class'] == 'Pathogenic').sum()}")

Wrote 17919 mutant ORF sequences to 'MAVEable_missense_variants.fasta'
  Benign    : 8956
  Pathogenic: 8963


In [29]:
# ── Wild-type rows ────────────────────────────────────────────────────────────
wt = (MAVEable_filtered[['GeneSymbol', 'HGNC_ID', 'mane_enst', 'Entry', 'orf_sequence']]
      .dropna(subset=['orf_sequence'])
      .rename(columns={'orf_sequence': 'sequence'})
      .assign(p_variant='', cdna_change='', classification='WT'))

# ── Missense variant rows ─────────────────────────────────────────────────────
gene_to_entry = MAVEable_filtered.set_index('GeneSymbol')['Entry'].to_dict()

missense = (variants_df[['GeneSymbol', 'HGNC_ID', 'mane_enst', 'p_variant', 'cdna_change', 'class', 'mutant_seq']]
            .rename(columns={'mutant_seq': 'sequence', 'class': 'classification'}))
missense['Entry'] = missense['GeneSymbol'].map(gene_to_entry)

# ── Combine ───────────────────────────────────────────────────────────────────
col_order = ['GeneSymbol', 'HGNC_ID', 'mane_enst', 'Entry',
             'p_variant', 'cdna_change', 'classification', 'sequence']

combined = pd.concat([wt[col_order], missense[col_order]], ignore_index=True)

csv_out = "MAVEable_sequences.csv"
combined.to_csv(csv_out, index=False)

print(f"Wrote {len(combined):,} rows to '{csv_out}'")
print(combined.groupby('classification').size().to_string())
combined.head()

Wrote 20,119 rows to 'MAVEable_sequences.csv'
classification
Benign        8956
Pathogenic    8963
WT            2200


,GeneSymbol,HGNC_ID,mane_enst,Entry,p_variant,cdna_change,classification,sequence
0,AAAS,HGNC:13666,ENST00000209873,Q9NRG9,,,WT,ATGTGCTCTCTGGGGTTGTTCCCTCCTCCACCGCCTCGGGGTCAAG...
1,AAGAB,HGNC:25662,ENST00000261880,Q6PD74,,,WT,ATGGCTGCTGGCGTACCCTGTGCGTTAGTCACCAGCTGCTCCTCCG...
2,AARS1,HGNC:20,ENST00000261772,P49588,,,WT,ATGGACTCTACTCTAACAGCAAGTGAAATCCGGCAGCGATTTATAG...
3,AASS,HGNC:17366,ENST00000417368,Q9UDR5,,,WT,ATGCTGCAAGTACATAGGACTGGACTGGGCAGGCTGGGGGTCAGCC...
4,ABCA12,HGNC:14637,ENST00000272895,Q86UK0,,,WT,ATGGCTTCCCTGTTTCATCAGCTTCAGATCCTGGTCTGGAAAAATT...
